<a href="https://colab.research.google.com/github/Anaghasadpewar/CHATBOT/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#install dependency

! pip install -q groq ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 67.1 MB/s eta 0:00:00


In [2]:
# API key integrate

import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
 os.environ["GROQ_API_KEY"]=getpass("enter your api key: ")

 print("API key set."if os.environ.get("GROQ_API_KEY")else "no API key found")

enter your api key: ··········
API key set.


In [3]:
from groq import Groq
client = Groq()
for m in client.models.list().data:
  print(m.id)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo


In [4]:
# backend

from groq import Groq
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Chatbot:
  model : str = "openai/gpt-oss-20b"
  system_prompt: str = "You are a helpful, concised assistent."
  temperature: float = 0.7
  # temperature controls the behaviour of output if less value then creaative output , if more then mature output (0 to 1)
  max_tokens: int =1024  #sets limit of words

  _client: Groq = field(default=None, repr=False) #maintaining connection with client
  history: List[Dict[str,str]]= field(default_factory=list, repr = False)

  def __post_init__(self): #initialising client
    self._client = Groq() #reads GROQ_API_KEY from environment

  def send(self, user_message: str) -> str:
    self.history.append({"role":"user", "content": user_message})
    messages = [{"role":"user", "content":self.system_prompt},*self.history]
    try:#exception handling
      response = self._client.chat.completions.create(
          model= self.model,
          messages=messages,
          temperature=self.temperature,
          max_tokens=self.max_tokens

      )
    except Exception as e:
      self.history.pop()  #remove recent error
      raise RuntimeError(f"Groq API error:{e}") from e

    # reply to user
    reply= response.choices[0].message.content
    self.history.append({"role":"user","content":reply})
    return reply


In [5]:
bot = Chatbot()
bot_msg = bot.send("tell me abount AI")
print(bot_msg)

Artificial Intelligence (AI) is the branch of computer science that builds systems capable of performing tasks that normally require human intelligence. It includes:

- **Machine Learning** – algorithms that learn patterns from data (e.g., image recognition, natural language processing).
- **Deep Learning** – neural‑network models with many layers that excel at complex pattern recognition.
- **Reinforcement Learning** – agents that learn by trial‑and‑error interactions with an environment.
- **Rule‑based & symbolic AI** – knowledge‑based systems that use explicit logic and rules.

Key applications: speech and image recognition, autonomous vehicles, recommendation engines, medical diagnosis, and chatbots. AI research balances performance, interpretability, and ethical considerations such as bias, privacy, and safety.


In [6]:
import ipywidgets as widgets
from IPython.display import display, HTML
import html
import traceback

bot = Chatbot(model="openai/gpt-oss-20b")   # default

model_dropdown = widgets.Dropdown(
    options=["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b", "allam-2-7b"],
    value=bot.model,
    desdescriptioncription="Model:",
    layout=widgets.Layout(width="300px"),
)

temp_slider = widgets.FloatSlider(
    value=bot.temperature, min=0.0, max=1.0, step=0.1,
    description="Temp:", continuous_update=False,
    layout=widgets.Layout(width="300px"),
)

chat_log = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", height="400px", overflow_y="auto", padding="8px")
)
text_input = widgets.Text(placeholder="Type a message and press Enter...", layout=widgets.Layout(width="80%"))
send_button = widgets.Button(description="Send", button_style="primary")
clear_button = widgets.Button(description="Clear chat", button_style="warning")
status_label = widgets.Label(value="")

input_row = widgets.HBox([text_input, send_button, clear_button])
controls_row = widgets.HBox([model_dropdown, temp_slider])
ui = widgets.VBox([controls_row, chat_log, input_row, status_label])


def render_bubble(role, text):
    if role == "user":
        bg, color, align = "#DCF8C6", "#000000", "right"
    elif role == "assistant":
        bg, color, align = "#F1F0F0", "#000000", "left"
    else:  # error
        bg, color, align = "#FFD6D6", "#7A0000", "left"

    safe_text = html.escape(text).replace("\n", "<br>")
    bubble = f"""
    <div style="text-align:{align}; margin:6px 0;">
      <span style="display:inline-block; background:{bg}; color:{color}; padding:8px 12px;
                    border-radius:10px; max-width:75%; text-align:left;">
        <b>{role}:</b><br>{safe_text}
      </span>
    </div>
    """
    with chat_log:
        display(HTML(bubble))


def on_send(_=None):
    message = text_input.value.strip()
    if not message:
        return
    text_input.value = ""
    text_input.disabled = send_button.disabled = True
    status_label.value = "Waiting for response..."
    render_bubble("user", message)

    bot.model = model_dropdown.value
    bot.temperature = temp_slider.value

    try:
        reply = bot.send(message)
        render_bubble("assistant", reply)
        status_label.value = ""
    except Exception as e:
        render_bubble("error", str(e))
        status_label.value = "Error — see message above."
        traceback.print_exc()
    finally:
        text_input.disabled = send_button.disabled = False


def on_clear(_=None):
    bot.reset()
    chat_log.clear_output()
    status_label.value = "Chat cleared."


send_button.on_click(on_send)
clear_button.on_click(on_clear)
text_input.on_submit(on_send)

display(ui)